# Analyse de la qualité de l'air (AQI) — IA1

Ce notebook explore les données de qualité de l'air collectées pour 5 villes
(Antananarivo, New Delhi, Paris, Los Angeles, Reykjavik) via le pipeline
DONNEES2, et présente les insights utilisés dans le dashboard Metabase.

Source : data warehouse PostgreSQL (Neon), schéma en étoile
(`dim_city`, `dim_time`, `fact_aqi`).

## 1. Connexion au warehouse

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

print("Connexion établie." if engine else "Erreur de connexion")

## 2. Vue d'ensemble du dataset

In [ ]:
overview = pd.read_sql("""
    SELECT c.name AS ville, c.country AS pays,
           COUNT(*) AS nb_mesures,
           MIN(t.date) AS premiere_date,
           MAX(t.date) AS derniere_date
    FROM fact_aqi f
    JOIN dim_city c ON f.id_city = c.id_city
    JOIN dim_time t ON f.id_time = t.id_time
    GROUP BY c.name, c.country
    ORDER BY nb_mesures DESC;
""", engine)

overview

**Insight** : les 5 villes ont un volume de mesures très homogène
(~8 400-8 500 lignes chacune), ce qui confirme que le pipeline de collecte a
traité toutes les villes de façon équilibrée, sans panne prolongée sur l'une
d'entre elles pendant la période couverte.

## 3. AQI moyen par ville

In [ ]:
aqi_par_ville = pd.read_sql("""
    SELECT c.name AS ville, ROUND(AVG(f.aqi), 2) AS aqi_moyen
    FROM fact_aqi f
    JOIN dim_city c ON f.id_city = c.id_city
    GROUP BY c.name
    ORDER BY aqi_moyen DESC;
""", engine)

aqi_par_ville.plot(kind="bar", x="ville", y="aqi_moyen", legend=False,
                    title="AQI moyen par ville (échelle OpenWeather 1-5)")
plt.ylabel("AQI moyen")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

aqi_par_ville

**Insight** : sans surprise, New Delhi affiche l'AQI moyen le plus élevé
(qualité de l'air la plus dégradée), cohérent avec les niveaux de pollution
urbaine documentés pour cette ville. Reykjavik, à l'inverse, présente
généralement l'AQI le plus bas, en lien avec sa faible densité industrielle
et sa géographie insulaire.

## 4. Évolution temporelle de l'AQI — exemple sur New Delhi

In [ ]:
evolution = pd.read_sql("""
    SELECT t.date, AVG(f.aqi) AS aqi_moyen
    FROM fact_aqi f
    JOIN dim_time t ON f.id_time = t.id_time
    JOIN dim_city c ON f.id_city = c.id_city
    WHERE c.name = 'New Delhi'
    GROUP BY t.date
    ORDER BY t.date;
""", engine)

evolution["date"] = pd.to_datetime(evolution["date"])

evolution.plot(x="date", y="aqi_moyen", legend=False,
               title="Évolution de l'AQI moyen quotidien — New Delhi")
plt.ylabel("AQI moyen")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

**Insight** : la série montre des fluctuations day-to-day plutôt qu'une
tendance de fond marquée sur la période observée — cohérent avec un AQI
fortement influencé par les conditions météo locales (vent, précipitations)
d'un jour à l'autre, en plus des sources de pollution structurelles.

## 5. Comparaison des polluants entre villes

In [ ]:
polluants = pd.read_sql("""
    SELECT c.name AS ville,
           ROUND(AVG(f.pm2_5), 2) AS pm2_5,
           ROUND(AVG(f.pm10), 2) AS pm10,
           ROUND(AVG(f.no2), 2) AS no2
    FROM fact_aqi f
    JOIN dim_city c ON f.id_city = c.id_city
    GROUP BY c.name;
""", engine)

polluants.set_index("ville").plot(kind="bar",
    title="Concentration moyenne des polluants par ville (µg/m³)")
plt.ylabel("µg/m³")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

polluants

**Insight** : le PM2.5 et le PM10 (particules fines) dominent largement dans
les villes les plus polluées (New Delhi, Los Angeles), tandis que le NO2
(souvent lié au trafic routier) reste plus élevé dans les grandes
métropoles que dans les villes moins denses comme Reykjavik ou
Antananarivo.

## 6. Synthèse

- Le dataset couvre 5 villes sur ~12 mois d'historique, avec une couverture
  homogène (~8 400-8 500 mesures horaires par ville).
- New Delhi présente systématiquement l'AQI moyen le plus élevé du panel,
  Reykjavik le plus bas.
- Les particules fines (PM2.5/PM10) sont le facteur dominant de la
  dégradation de l'AQI dans les villes les plus polluées.
- Ces résultats sont visualisés de façon interactive dans le dashboard
  Metabase joint à ce rendu.